# MAPLE event selection (cafpyana)

Loads a maple `.df` (produced with `analysis_village/maple/configs/maple_evt*.py`),
applies/inspects the MAPLE 1muNp cut chain, and builds the sBruce-equivalent
variables and truth-level efficiency (`Pass_cut`) exactly as CAFANA-MAPLE does.

Run from the cafpyana root after `source setup.sh`.

In [ ]:
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# path to a maple .df (no-selection mode recommended, so the cutflow is visible)
DF = "/exp/sbnd/app/users/gputnam/osc2/MAPLE/validation-out/maple_run4val_cafanapid.df"

def load_df(fname, table):
    with h5py.File(fname, "r") as f:
        keys = [k for k in f.keys() if k.rsplit("_", 1)[0] == table]
    if not keys: return None
    return pd.concat([pd.read_hdf(fname, k) for k in sorted(keys, key=lambda k: int(k.rsplit("_",1)[1]))])

evt = load_df(DF, "evt")
mcnu = load_df(DF, "mcnu")
hdr = load_df(DF, "hdr")
print(len(evt), "slices,", len(mcnu) if mcnu is not None else 0, "true nus,", len(hdr), "events")
print("PID mode of this production:", evt.pid_mode.iloc[0])

## Cutflow

In [ ]:
cuts = ["cut_sanity", "cut_fv", "cut_crtveto", "cut_cryo", "cut_contained",
        "cut_muon", "cut_np", "cut_0pi", "cut_0shwother"]
labels = ["sanity", "FV", "CRT veto", "cryo-light", "containment",
          "1 muon", "N>1 protons", "0 pions", "0 showers/other"]
mask = pd.Series(True, index=evt.index)
print("%-18s %8s" % ("cut", "slices"))
print("%-18s %8d" % ("all", len(evt)))
for c, l in zip(cuts, labels):
    mask &= evt[c]
    print("%-18s %8d" % (l, mask.sum()))
sel = evt[evt.maple_sel]
print("selected (maple_sel):", len(sel))

## sBruce variables for selected slices

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(15, 8))
sel = evt[evt.maple_sel]
axs[0,0].hist(sel.recoE, bins=np.linspace(0, 3, 31)); axs[0,0].set_xlabel("recoE [GeV]")
axs[0,1].hist(sel.Muon_length, bins=np.linspace(0, 400, 41)); axs[0,1].set_xlabel("Muon length [cm]")
axs[0,2].hist(sel.Proton_length_leading, bins=np.linspace(0, 100, 41)); axs[0,2].set_xlabel("Leading proton length [cm]")
axs[1,0].hist(sel.deltaPt, bins=np.linspace(0, 1, 26)); axs[1,0].set_xlabel(r"$\delta p_T$ [GeV]")
axs[1,1].hist(sel.T3D_angle_mup, bins=np.linspace(-1, 1, 26)); axs[1,1].set_xlabel(r"cos $\theta_{\mu p}$")
axs[1,2].hist(sel.Number_protons, bins=np.arange(0.5, 8)); axs[1,2].set_xlabel("N protons")
fig.tight_layout()

## PID: cafpyana-recomputed (gump-style) vs CAFANA-compat chi2

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
mu = evt[evt.cut_muon]
axs[0].scatter(mu.Muon_chi2mu_cafana, mu.Muon_chi2mu_cafpyana, s=2)
axs[0].plot([0, 120], [0, 120], "r--"); axs[0].set_xlabel(r"$\chi^2_\mu$ (cafana)"); axs[0].set_ylabel(r"$\chi^2_\mu$ (cafpyana)")
axs[1].scatter(mu.Muon_chi2pro_cafana, mu.Muon_chi2pro_cafpyana, s=2)
axs[1].plot([0, 400], [0, 400], "r--"); axs[1].set_xlabel(r"$\chi^2_p$ (cafana)"); axs[1].set_ylabel(r"$\chi^2_p$ (cafpyana)")
fig.tight_layout()

print("selection agreement between PID flavors:")
print("  primary:", evt.maple_sel.sum(), " alt:", evt.maple_sel_alt.sum(),
      " both:", (evt.maple_sel & evt.maple_sel_alt).sum())

## Truth-level efficiency (`Pass_cut`)

Replicates CAFANA `kEff_cuts`: for each true 1muNp interaction, the maximum
cut level reached by any truth-matched slice (tmatch eff >= 0.5).

In [ ]:
nus = mcnu[mcnu.is_1muNp_maple].copy()
# match slices to true interactions: same event, tmatch idx == nu index, tmatch eff >= 0.5
evt_eff = evt[~(evt.tmatch_eff < 0.5)].reset_index()
key = [c for c in ("__ntuple", "entry") if c in evt_eff.columns]
evt_eff["tidx"] = evt_eff.tmatch_idx.astype(int)
maxcut = evt_eff.groupby(key + ["tidx"]).maxcut.max()

n = nus.reset_index()
lookup = pd.MultiIndex.from_arrays([n[k] for k in key] + [n["ind"].astype(int)])
n["Pass_cut"] = maxcut.reindex(lookup).fillna(1).clip(lower=1).values
print(n.Pass_cut.value_counts().sort_index())

fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
axs[0].hist(n.Pass_cut, bins=np.arange(0.5, 12)); axs[0].set_xlabel("Pass_cut (max cut reached)")
enu_bins = np.linspace(0, 3, 16)
den, _ = np.histogram(n.nu_E, bins=enu_bins)
num, _ = np.histogram(n.nu_E[n.Pass_cut == 10], bins=enu_bins)
ctr = (enu_bins[:-1] + enu_bins[1:]) / 2
with np.errstate(invalid="ignore"):
    axs[1].step(ctr, num / den, where="mid")
axs[1].set_xlabel("true E$_\\nu$ [GeV]"); axs[1].set_ylabel("selection efficiency")
fig.tight_layout()

## Calorimetric variations (chi2 of the muon candidate)

In [ ]:
calovars = [c.replace("Muon_chi2mu_", "") for c in evt.columns
            if c.startswith("Muon_chi2mu_") and c.split("_")[-1] not in ("cafpyana", "cafana")]
if calovars:
    mu = evt[evt.maple_sel]
    print("%-12s %10s %10s" % ("variation", "mean chi2mu", "mean chi2p"))
    print("%-12s %10.2f %10.2f" % ("nominal", mu.Muon_chi2mu_cafpyana.mean(), mu.Muon_chi2pro_cafpyana.mean()))
    for v in calovars:
        print("%-12s %10.2f %10.2f" % (v, mu["Muon_chi2mu_%s" % v].mean(), mu["Muon_chi2pro_%s" % v].mean()))
else:
    print("this production was made with do_calo_syst=False")